In [2]:
import numpy as np
import torch
from motion_process import recover_from_ric

motion = np.load("2026-03-09-07_09_0742399.npy")   # 달리기
# motion = np.load("2026-03-09-07_09_4951239.npy")   # 점프
xyz = recover_from_ric(x, 22).squeeze(0).cpu().numpy()

# 루트 관절만 몇 프레임 출력
print("ROOT joint XYZ (첫 10프레임):")
for t in range(10):
    print(f"  frame {t}: {xyz[t, 0]}")

# 수직축이 뭔지 확인 - 가장 변화 적은 축이 height
print("\nRMS per axis (root):")
root = xyz[:, 0, :]
print(f"  X std: {root[:,0].std():.4f}")
print(f"  Y std: {root[:,1].std():.4f}")
print(f"  Z std: {root[:,2].std():.4f}")

ROOT joint XYZ (첫 10프레임):
  frame 0: [0.       0.833897 0.      ]
  frame 1: [0.00828545 0.8137877  0.06540497]
  frame 2: [0.01205628 0.8078577  0.13913962]
  frame 3: [0.01586702 0.8079771  0.21624061]
  frame 4: [0.0193183  0.80827516 0.29509276]
  frame 5: [0.0223225 0.8165737 0.3808996]
  frame 6: [0.02489133 0.82486165 0.4761691 ]
  frame 7: [0.02687602 0.82569826 0.5834637 ]
  frame 8: [0.02840287 0.8301914  0.7041516 ]
  frame 9: [0.02949107 0.831959   0.8376553 ]

RMS per axis (root):
  X std: 0.4953
  Y std: 0.0693
  Z std: 2.5363


In [33]:
import numpy as np
import torch
from motion_process import recover_from_ric

motion = np.load("2026-03-09-07_09_0742399.npy")   # (T,263)
# motion = np.load("2026-03-09-07_09_4951239.npy")   # (T,263)

x = torch.from_numpy(motion).float().unsqueeze(0)

xyz = recover_from_ric(x, 22)

print(xyz.shape)

torch.Size([1, 60, 22, 3])


In [34]:
xyz = xyz.squeeze(0).cpu().numpy()   # (88, 22, 3)
print(xyz.shape)

(60, 22, 3)


In [35]:
print("min:", xyz.min(), "max:", xyz.max())
print("root first 5 frames:\n", xyz[:5, 0, :])

min: -0.523127 max: 8.768153
root first 5 frames:
 [[0.         0.833897   0.        ]
 [0.00828545 0.8137877  0.06540497]
 [0.01205628 0.8078577  0.13913962]
 [0.01586702 0.8079771  0.21624061]
 [0.0193183  0.80827516 0.29509276]]


In [36]:
JOINTS = [
    "root",
    "l_hip", "r_hip",
    "spine",
    "l_knee", "r_knee",
    "spine2",
    "l_ankle", "r_ankle",
    "spine3",
    "l_foot", "r_foot",
    "neck",
    "l_collar", "r_collar",
    "head",
    "l_shoulder", "r_shoulder",
    "l_elbow", "r_elbow",
    "l_wrist", "r_wrist",
]

In [ ]:
import json
from pathlib import Path
import numpy as np

# HumanML3D 22-joint indices
ROOT = 0
L_HIP, R_HIP = 1, 2
SPINE = 3
L_KNEE, R_KNEE = 4, 5
SPINE2 = 6
L_ANKLE, R_ANKLE = 7, 8
SPINE3 = 9
L_FOOT, R_FOOT = 10, 11
NECK = 12
L_COLLAR, R_COLLAR = 13, 14
HEAD = 15
L_SHOULDER, R_SHOULDER = 16, 17
L_ELBOW, R_ELBOW = 18, 19
L_WRIST, R_WRIST = 20, 21


def normalize(v, eps=1e-8):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.clip(n, eps, None)


def angle_wrap(a):
    """wrap to [-pi, pi]"""
    return (a + np.pi) % (2 * np.pi) - np.pi


def smooth_signal(x, win=5):
    if win <= 1:
        return x
    pad = win // 2
    x_pad = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(win, dtype=np.float32) / win
    return np.convolve(x_pad, kernel, mode="valid")


def smooth_array(arr, win=5):
    arr = np.asarray(arr)
    if arr.ndim == 1:
        return smooth_signal(arr, win)
    out = np.zeros_like(arr, dtype=np.float32)
    for i in range(arr.shape[1]):
        out[:, i] = smooth_signal(arr[:, i], win)
    return out


def vec_to_yaw_pitch(v):
    """
    assumes:
      x = left/right
      y = up/down
      z = forward/back
    returns radians
    """
    x = v[..., 0]
    y = v[..., 1]
    z = v[..., 2]
    yaw = np.arctan2(x, z)
    pitch = np.arctan2(y, np.sqrt(x * x + z * z))
    return yaw, pitch


def signed_angle_around_axis(a, b, axis):
    """
    signed angle from a -> b around axis
    all (...,3)
    """
    a_n = normalize(a)
    b_n = normalize(b)
    axis_n = normalize(axis)
    cross = np.cross(a_n, b_n)
    sinv = np.sum(cross * axis_n, axis=-1)
    cosv = np.sum(a_n * b_n, axis=-1)
    return np.arctan2(sinv, cosv)


def build_body_frame(xyz):
    """
    Build approximate body local axes from hips + neck.
    returns:
      right, up, forward  each (T,3)
    """
    hip_center = 0.5 * (xyz[:, L_HIP] + xyz[:, R_HIP])
    shoulder_center = 0.5 * (xyz[:, L_SHOULDER] + xyz[:, R_SHOULDER])

    right = normalize(xyz[:, R_HIP] - xyz[:, L_HIP])  # body right
    up = normalize(shoulder_center - hip_center)

    # forward = right x up (or up x right depending on handedness)
    forward = normalize(np.cross(right, up))
    # re-orthogonalize up
    up = normalize(np.cross(forward, right))
    return right, up, forward


def extract_steve_motion(
    xyz,
    fps=20,
    pos_scale=16.0,
    smooth_win=5,
    keep_y=True,
):
    """
    xyz: (T,22,3)
    returns dict ready to dump as JSON
    """
    xyz = np.asarray(xyz, dtype=np.float32)
    T = xyz.shape[0]

    # body frame
    body_right, body_up, body_fwd = build_body_frame(xyz)

    # root / body position
    root = xyz[:, ROOT].copy()
    root0 = root[0].copy()

    body_pos = root - root0
    if not keep_y:
        body_pos[:, 1] = 0.0
    body_pos *= pos_scale

    # body orientation from forward/up
    body_yaw = np.arctan2(body_fwd[:, 0], body_fwd[:, 2])
    body_pitch = np.arctan2(body_fwd[:, 1], np.sqrt(body_fwd[:, 0] ** 2 + body_fwd[:, 2] ** 2))
    body_roll = np.arctan2(body_right[:, 1], np.sqrt(body_right[:, 0] ** 2 + body_right[:, 2] ** 2))

    # head direction
    neck_to_head = xyz[:, HEAD] - xyz[:, NECK]
    head_yaw_world, head_pitch_world = vec_to_yaw_pitch(neck_to_head)
    head_yaw = angle_wrap(head_yaw_world - body_yaw + np.pi)
    head_pitch = head_pitch_world - body_pitch

    # arms: shoulder -> elbow
    l_arm = xyz[:, L_ELBOW] - xyz[:, L_SHOULDER]
    r_arm = xyz[:, R_ELBOW] - xyz[:, R_SHOULDER]

    # legs: hip -> knee
    l_leg = xyz[:, L_KNEE] - xyz[:, L_HIP]
    r_leg = xyz[:, R_KNEE] - xyz[:, R_HIP]

    # Express pitch relative to body forward axis.
    # Steve-style simplified rig: mostly pitch only.
    down_axis = -body_up

    left_arm_pitch = signed_angle_around_axis(body_fwd, l_arm, body_right)
    right_arm_pitch = signed_angle_around_axis(body_fwd, r_arm, body_right)

    left_leg_pitch = signed_angle_around_axis(down_axis, l_leg, body_right)
    right_leg_pitch = signed_angle_around_axis(down_axis, r_leg, body_right)

    # Optional secondary yaw for arms if you want later
    l_arm_yaw_world, _ = vec_to_yaw_pitch(l_arm)
    r_arm_yaw_world, _ = vec_to_yaw_pitch(r_arm)
    left_arm_yaw = angle_wrap(l_arm_yaw_world - body_yaw)
    right_arm_yaw = angle_wrap(r_arm_yaw_world - body_yaw)

    # smoothing
    body_pos = smooth_array(body_pos, smooth_win)
    body_yaw = smooth_signal(body_yaw, smooth_win)
    body_pitch = smooth_signal(body_pitch, smooth_win)
    body_roll = smooth_signal(body_roll, smooth_win)

    head_yaw = smooth_signal(head_yaw, smooth_win)
    head_pitch = smooth_signal(head_pitch, smooth_win)

    left_arm_pitch = smooth_signal(left_arm_pitch, smooth_win)
    right_arm_pitch = smooth_signal(right_arm_pitch, smooth_win)
    left_leg_pitch = smooth_signal(left_leg_pitch, smooth_win)
    right_leg_pitch = smooth_signal(right_leg_pitch, smooth_win)

    left_arm_yaw = smooth_signal(left_arm_yaw, smooth_win)
    right_arm_yaw = smooth_signal(right_arm_yaw, smooth_win)

    frames = []
    for i in range(T):
        frames.append({
            "frame": int(i),
            "time": float(i / fps),

            "body_pos": {
                "x": float(body_pos[i, 0]),
                "y": float(body_pos[i, 1]),
                "z": float(body_pos[i, 2]),
            },
            "body_rot": {
                "yaw": float(body_yaw[i]),
                "pitch": float(body_pitch[i]),
                "roll": float(body_roll[i]),
            },

            "head_rot": {
                "yaw": float(head_yaw[i]),
                "pitch": float(head_pitch[i]),
            },

            "left_arm_rot": {
                "pitch": float(left_arm_pitch[i]),
                "yaw": float(left_arm_yaw[i]),
            },
            "right_arm_rot": {
                "pitch": float(right_arm_pitch[i]),
                "yaw": float(right_arm_yaw[i]),
            },

            "left_leg_rot": {
                "pitch": float(left_leg_pitch[i]),
            },
            "right_leg_rot": {
                "pitch": float(right_leg_pitch[i]),
            },
        })

    return {
        "meta": {
            "source_format": "HumanML3D xyz",
            "target": "Steve intermediate",
            "fps": fps,
            "num_frames": T,
            "angles": "radians",
            "pos_scale": pos_scale,
        },
        "frames": frames,
    }


def rad_to_deg_motion(data):
    data = json.loads(json.dumps(data))  # deep copy
    for fr in data["frames"]:
        for grp in ["body_rot", "head_rot", "left_arm_rot", "right_arm_rot", "left_leg_rot", "right_leg_rot"]:
            for k, v in fr[grp].items():
                fr[grp][k] = float(np.degrees(v))
    data["meta"]["angles"] = "degrees"
    return data


def save_json(data, path):
    path = Path(path)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved:", path)

In [38]:
# xyz: (T,22,3)
data = extract_steve_motion(
    xyz,
    fps=20,        # 필요시 수정
    pos_scale=16,  # 마인크래프트 블록 느낌으로 스케일 조절
    smooth_win=5,
    keep_y=True,
)

data_deg = rad_to_deg_motion(data)
save_json(data_deg, "steve_motion_intermediate.json")

# 앞 3프레임 보기
for fr in data_deg["frames"][:3]:
    print(fr)

saved: steve_motion_intermediate.json
{'frame': 0, 'time': 0.0, 'body_pos': {'x': 0.0650935173034668, 'y': -0.1476755142211914, 'z': 0.6545426845550537}, 'body_rot': {'yaw': 169.03044799434906, 'pitch': 30.21360475583445, 'roll': -5.650830651310404}, 'head_rot': {'yaw': -155.83293791399697, 'pitch': 17.25445120204289}, 'left_arm_rot': {'pitch': 63.80053065971148, 'yaw': 5.705372273153895}, 'right_arm_rot': {'pitch': 96.24493325508004, 'yaw': 26.02508003978334}, 'left_leg_rot': {'pitch': 71.990419248177}, 'right_leg_rot': {'pitch': -2.266376935250941}}
{'frame': 1, 'time': 0.05, 'body_pos': {'x': 0.11586800217628479, 'y': -0.23061925172805786, 'z': 1.3465126752853394}, 'body_rot': {'yaw': 168.18738408471603, 'pitch': 32.24660714719416, 'roll': -5.801397048921458}, 'head_rot': {'yaw': -155.91517339160393, 'pitch': 13.713367801380107}, 'left_arm_rot': {'pitch': 64.58309958386569, 'yaw': 6.424036239886174}, 'right_arm_rot': {'pitch': 93.2902412314567, 'yaw': 24.988858380422364}, 'left_leg_

In [41]:
import json
import numpy as np

# --- [사용자 맞춤형 보정 파라미터] ---
move_mult = 0.0       # 0.0이면 제자리 고정 (이동량 완전 제거)
y_offset = 12.0       # 지면 높이 (발 위치 보정)
body_pitch_fix = -90.0 # 스티브 세우기 (-90 또는 90)
# ----------------------------------

def get_bend_angle(v1, v2):
    """관절 굽힘(Bend) 계산: 직선일 때 0, 굽힐수록 커짐"""
    v1_u = v1 / (np.linalg.norm(v1) + 1e-8)
    v2_u = v2 / (np.linalg.norm(v2) + 1e-8)
    angle = np.degrees(np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0)))
    return round(max(0, 180 - angle), 5)

def build_steve_202_miframes(xyz_raw, output_file):
    # xyz_raw: (T, 22, 3) 
    T = xyz_raw.shape[0]
    
    # 0프레임 위치 (제자리 고정용)
    start_pos = xyz_raw[0, 0].copy()

    miframes = {
        "format": 34, "created_in": "2.0.2", "is_model": True,
        "tempo": 20, "length": T, "keyframes": []
    }

    # HumanML3D 관절 인덱스 매핑
    # 0:Root, 12:Neck, 15:Head, 16:L_Shoulder, 18:L_Elbow, 20:L_Wrist ...
    parts = {
        "head": {"p": 15, "parent": 12},
        "left_arm": {"p": 18, "parent": 16, "end": 20},
        "right_arm": {"p": 19, "parent": 17, "end": 21},
        "left_leg": {"p": 4, "parent": 1, "end": 7},
        "right_leg": {"p": 5, "parent": 2, "end": 8}
    }

    for t in range(T):
        curr_xyz = xyz_raw[t]
        
        # 1. Root / Body (위치 및 전신 회전)
        # Mine-imator 축: Y=Up, Z=Forward
        root_pos = curr_xyz[0] - start_pos
        root_vals = {
            "POS_X": round(root_pos[0] * 16 * move_mult, 5),
            "POS_Y": round(root_pos[1] * 16 + y_offset, 5), # 높이는 그대로 유지
            "POS_Z": round(root_pos[2] * 16 * move_mult, 5),
            "ROT_X": body_pitch_fix, # 전신 회전 고정
            "ROT_Y": 0, "ROT_Z": 0
        }
        miframes["keyframes"].append({"position": t, "values": root_vals})

        # 2. 각 파트별 회전 및 BEND 적용
        for name, idx in parts.items():
            # 벡터 계산
            v = curr_xyz[idx['p']] - curr_xyz[idx['parent']]
            vx, vy, vz = v[0], v[1], v[2]
            
            # 사용자 정의 축 매핑 (다윈 인체비례도 기반)
            # Y: 옆구리~좌우나란히 (vx, vy 기준)
            # X: 앞뒤 흔들기 (vy, vz 기준)
            # Z: 앞으로 오므리기 (vx, vz 기준)
            rot_y = np.degrees(np.arctan2(vx, -vy + 1e-8))
            rot_x = np.degrees(np.arctan2(vz, np.sqrt(vx**2 + vy**2) + 1e-8))
            rot_z = np.degrees(np.arctan2(vx, vz + 1e-8)) # Z축(오므리기) 추가

            vals = {
                "ROT_X": round(rot_x, 5),
                "ROT_Y": round(rot_y, 5),
                "ROT_Z": round(rot_z, 5) # Z축 드디어 적용
            }

            # 팔/다리 관절 굽힘(BEND) 추가
            if 'end' in idx:
                v1 = curr_xyz[idx['parent']] - curr_xyz[idx['p']]
                v2 = curr_xyz[idx['end']] - curr_xyz[idx['p']]
                vals["BEND_ANGLE_X"] = get_bend_angle(v1, v2)

            miframes["keyframes"].append({
                "position": t, 
                "part_name": name, 
                "values": vals
            })

    with open(output_file, 'w', encoding='utf-8') as out:
        json.dump(miframes, out, indent="\t")
    print(f"✅ 최종 본 완성! '{output_file}'를 Mine-imator 2.0.2에서 불러오세요.")

# 실행 (노트북 환경의 xyz 변수 사용)
build_steve_202_miframes(xyz.squeeze(), 'steve_perfect_202.miframes')

✅ 최종 본 완성! 'steve_perfect_202.miframes'를 Mine-imator 2.0.2에서 불러오세요.
